# Unified Ghost-Frequency Falsification Suite

Three competing hypotheses for the observed 20-40 kHz quasi-static detuning
(Kingston: -1.130 rad / 4.392 us; Fez N21: +0.59 rad / 4.75 us):

| Hypothesis | Claim | Signature |
|---|---|---|
| **H0 (null)** | Standard noise model; no detuning | Flat phase vs delay |
| **H1 (intrinsic)** | Quasi-static 30 kHz qubit detuning | Constant drift; pre-delay independent |
| **H2 (photonic)** | AC Stark from residual resonator photons (2χn) | Drift DECAYS with pre-delay |

This suite tries to kill each one. Whatever survives earns QPU time.


In [ ]:
import math

import numpy as np
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))
torch.manual_seed(7)

# ---- Physical parameters ----
T2_star_us = 80.0
chi_khz = 1000.0
nbar0 = 0.015
T1r_us = 5.0
intrinsic_khz = 30.0
t_ramsey_us = 4.4
print(f"H2 Stark at tau=0: {2 * chi_khz * nbar0:.1f} kHz")

## Test 1 -- Null vs H1 vs H2 at the observed idle times
Do all three hypotheses reproduce (or correctly fail to reproduce) the anchors?


In [ ]:
def phase_H0(t_us):
    return 0.0


def phase_H1(t_us, d=intrinsic_khz):
    return 2 * math.pi * d * 1e3 * t_us * 1e-6


def phase_H2(tau_us, t_us):
    tau, t, T1r = tau_us * 1e-6, t_us * 1e-6, T1r_us * 1e-6
    integ = 2 * chi_khz * 1e3 * nbar0 * T1r * (math.exp(-tau / T1r) - math.exp(-(tau + t) / T1r))
    return 2 * math.pi * integ


print(f"{'t (us)':>7} {'obs (rad)':>10} {'H0':>8} {'H1':>8} {'H2 tau=0':>9}")
for t, obs in [(4.392, -1.130), (4.75, 0.59)]:
    print(f"{t:>7.3f} {obs:>10.3f} {phase_H0(t):>8.3f} {phase_H1(t):>8.3f} {phase_H2(0, t):>9.3f}")
print()
print("H0 predicts 0 (fails to explain). H1 and H2 both match magnitude.")

## Test 2 -- The pre-delay discriminator (H1 vs H2)
Vary τ between init and Ramsey. H1: flat. H2: exponential decay.


In [ ]:
def delta_eff(tau_us, model, n_shots=2000, t_us=t_ramsey_us):
    if model == "H1":
        ph = phase_H1(t_us)
    elif model == "H2":
        ph = phase_H2(tau_us, t_us)
    else:
        ph = 0.0
    p_true = 0.5 * (1 + math.cos(ph))
    p_hat = (torch.rand(n_shots, device=device) < p_true).float().mean().item()
    c = max(-1.0, min(1.0, 2 * p_hat - 1))
    return math.acos(c) / (2 * math.pi * t_us * 1e-6) / 1e3


taus = [0.0, 2.0, 5.0, 10.0, 20.0, 40.0]
print(f"{'tau':>6} {'H0':>8} {'H1':>8} {'H2':>8}")
for tau in taus:
    print(f"{tau:>6.1f} {delta_eff(tau, 'H0'):>8.1f} {delta_eff(tau, 'H1'):>8.1f} {delta_eff(tau, 'H2'):>8.1f}")

## Test 3 -- Power: shots to separate H1 from H0 at 5σ


In [ ]:
def sigma_phase(n_shots, t_us=8.0, d_khz=20.0, trials=1500):
    p = 0.5 * (1 + math.cos(2 * math.pi * d_khz * 1e3 * t_us * 1e-6))
    ph = torch.acos(torch.clamp(2 * (torch.rand(trials, n_shots, device=device) < p).float().mean(1) - 1, -1, 1))
    return ph.std().item()


tgt = 2 * math.pi * 20e3 * 8e-6
for n in [200, 500, 1000, 2000]:
    s = sigma_phase(n)
    print(f"N={n:5d}: {tgt / s:.1f} sigma")

## Test 4 -- Mimic attempts (trying to fake each hypothesis)
4a. Amplitude miscalibration -> can it fake H1/H2's drifting phase?  
4b. Static ZZ -> can it fake H1's spectator-independent drift?


In [ ]:
# 4a: miscalibrated X/2 (eps=0.2) -- visibility drops, phase stays flat
t = np.linspace(0, 8, 5)
vis = math.cos(0.2) ** 2
p_mis = 0.5 * (1 + vis * np.cos(0 * t))
p_h1 = 0.5 * (1 + np.cos(2 * np.pi * 30e3 * t * 1e-6))
print("t:      ", " ".join(f"{x:.1f}" for x in t))
print("miscal: ", " ".join(f"{x:.3f}" for x in p_mis))
print("H1:     ", " ".join(f"{x:.3f}" for x in p_h1))
print("-> miscalibration: flat. H1: oscillates. Cannot mimic.")
print()
# 4b: ZZ=25kHz spectator |1> vs H1 intrinsic
for tt in [2.0, 4.0, 8.0]:
    pzz0 = 0.5 * (1 + math.cos(0))
    pzz1 = 0.5 * (1 + math.cos(2 * math.pi * 25e3 * tt * 1e-6))
    ph1 = 0.5 * (1 + math.cos(2 * math.pi * 30e3 * tt * 1e-6))
    print(f"t={tt:.0f}us  ZZ|0>={pzz0:.3f} ZZ|1>={pzz1:.3f}  H1={ph1:.3f}")
print("-> ZZ is spectator-dependent; H1 is not. Separable by Experiment A.")

## Test 5 -- Active reset prediction (H2-specific)
If H2 is right, active resonator reset before Ramsey should erase the detuning.
If H1 is right, reset changes nothing.


In [ ]:
# H2 with perfect reset: nbar0 -> 0
d_no_reset = phase_H2(0, t_ramsey_us) / (2 * math.pi * t_ramsey_us * 1e-6) / 1e3
print(f"H2 detuning without reset: {d_no_reset:.1f} kHz")
print("H2 detuning with reset:    0.0 kHz")
print(f"H1 detuning with reset:    {intrinsic_khz:.1f} kHz (unchanged)")
print("-> Second independent discriminator for Claude Code's QPU run.")

## Suite verdict

| Falsifier | Kills H0? | Kills H1? | Kills H2? |
|---|---|---|---|
| Observed drift exists | yes | -- | -- |
| Pre-delay decay | -- | yes if decays | yes if flat |
| Active reset erases it | -- | yes if erased | yes if persists |
| Spectator \|0>/\|1> split | -- | yes if split | -- |

The QPU run (Claude Code) should execute in falsification order:
1. **Cheapest:** pre-delay sweep on one qubit (kills H1 or H2 immediately).
2. Active reset comparison (independent confirmation).
3. Full-chip checkerboard only if H1 survives 1-2.
